**Author:** Ashutosh Jayant  
**Project:** India State Competitiveness Index (ISCI)

# 13 - Scenario Analysis

**This notebook answers one research question.**

**Research Question:** How much could a state's score change if one priority area improved?

These scenarios are simulations. They do not predict what will happen in reality. They only
show what the index would look like if the selected score improved while everything else
stayed the same.

The Version 1.0 normalization scale is kept fixed. Only the selected state's score is
recalculated. Other states are unchanged.

## Setup

Load the scores, the ranking and the groups. Work out each state's main priority (its
biggest gap to the top group) and the top group's average for each number.

In [2]:
import sys
from pathlib import Path
import pandas as pd

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

reports_dir = root / "version2" / "reports"

scores = pd.read_csv(root / "results" / "indicator_scores.csv", index_col=0)
index = pd.read_csv(root / "results" / "competitiveness_index.csv", index_col=0)
clusters = pd.read_csv(reports_dir / "state_clusters.csv", index_col=0)["cluster"]
ranked = index[index["Rank"].notna()].index

top_group = clusters[clusters == "Strong on both parts"].index
top_group_avg = scores.loc[top_group].mean()
gap_top = top_group_avg - scores.loc[ranked]
main_priority = gap_top.idxmax(axis=1)

base_score = index.loc[ranked, "competitiveness_score"]

print("states:", len(ranked), "| top group:", len(top_group))
print("\nexample main priorities:")
print(main_priority.head(5).to_string())

states: 33 | top group: 12

example main priorities:
state
Goa             unemployment_rate
Tamil Nadu        investment_rate
Gujarat                       ger
Puducherry           road_density
Telangana     manufacturing_share


## Simulate closing the biggest gap

For each state, raise its main-priority indicator to the top-group average, keep the frozen
scale, and recompute its overall score and rank. Every other state stays the same.

In [3]:
def simulate(state):
    ind = main_priority[state]
    row = scores.loc[state].copy()
    row[ind] = top_group_avg[ind]          # raise the main-priority indicator to the top-group average
    new_score = row.mean()
    new_rank = 1 + int((base_score.drop(state) > new_score).sum())
    return ind, new_score, new_rank

rows = []
for s in ranked:
    ind, new_score, new_rank = simulate(s)
    old_rank = int(index.loc[s, "Rank"])
    rows.append({
        "old_rank": old_rank,
        "indicator": ind,
        "old_score": round(base_score[s], 3),
        "new_score": round(new_score, 3),
        "score_gain": round(new_score - base_score[s], 3),
        "new_rank": new_rank,
        "rank_gain": old_rank - new_rank,
    })
sweep = pd.DataFrame(rows, index=ranked).sort_values("rank_gain", ascending=False)
sweep.to_csv(reports_dir / "scenario_sweep.csv", index_label="state")

print(sweep.to_string())

                           old_rank            indicator  old_score  new_score  score_gain  new_rank  rank_gain
state                                                                                                          
Tripura                          25      investment_rate      0.320      0.374       0.054        19          6
Madhya Pradesh                   24      factory_density      0.326      0.379       0.053        18          6
Andaman & Nicobar Islands        29      factory_density      0.254      0.334       0.080        23          6
Manipur                          28  manufacturing_share      0.276      0.332       0.056        23          5
Mizoram                          23  manufacturing_share      0.331      0.390       0.059        18          5
Jammu & Kashmir                  26      factory_density      0.295      0.350       0.056        21          5
Chandigarh                       19      investment_rate      0.370      0.422       0.052        15    

## Case studies

Four states from different groups. For each: where it starts, which gap is closed, and where
it ends. Only the numbers, no predictions.

In [4]:
cases = ["Bihar", "Kerala", "Tamil Nadu", "Uttar Pradesh"]

for s in cases:
    ind, new_score, new_rank = simulate(s)
    old_rank = int(index.loc[s, "Rank"])
    print(f"{s} ({clusters[s]})")
    print(f"  before:     score {base_score[s]:.3f}, rank {old_rank}")
    print(f"  biggest gap: {ind}")
    print(f"  simulation: raise {ind} to the top-group average ({top_group_avg[ind]:.3f})")
    print(f"  after:      score {new_score:.3f}, rank {new_rank}")
    print(f"  change:     score +{new_score - base_score[s]:.3f}, rank {old_rank} -> {new_rank}")
    print()

Bihar (Weak on both parts)
  before:     score 0.245, rank 30
  biggest gap: factory_density
  simulation: raise factory_density to the top-group average (0.650)
  after:      score 0.304, rank 26
  change:     score +0.059, rank 30 -> 26

Kerala (Strong basic conditions, weaker industry)
  before:     score 0.417, rank 15
  biggest gap: investment_rate
  simulation: raise investment_rate to the top-group average (0.560)
  after:      score 0.458, rank 11
  change:     score +0.042, rank 15 -> 11

Tamil Nadu (Strong on both parts)
  before:     score 0.605, rank 2
  biggest gap: investment_rate
  simulation: raise investment_rate to the top-group average (0.560)
  after:      score 0.613, rank 1
  change:     score +0.009, rank 2 -> 1

Uttar Pradesh (Weak on both parts)
  before:     score 0.280, rank 27
  biggest gap: factory_density
  simulation: raise factory_density to the top-group average (0.650)
  after:      score 0.327, rank 24
  change:     score +0.047, rank 27 -> 24

